# 12주차 실습: Spark 파티션·캐시·조인 실험

강의: [PDF](../pdf/week12_spark_optimization.pdf) · **PDF 실제 페이지 4–34** (인쇄 번호와 다를 수 있음)

## 학습 목표
동일한 결과를 유지하며 실행 계획과 반복 측정으로 튜닝 효과를 비교한다.

권장 구성: 개념 확인 10분 → 코드 실행 30분 → 변경 실험 30분 → 결과 토의 20분.
설치는 README.md를 따릅니다. 새 커널에서 위에서 아래로 실행하세요.

In [ ]:
from pathlib import Path
import sys
here = Path.cwd().resolve()
root = next((p for p in [here, *here.parents] if (p / 'labkit').is_dir()), None)
if root is None:
    candidate = here / '강의자료' / '실습코드'
    if candidate.is_dir(): root = candidate
if root is None:
    raise RuntimeError('실습코드 폴더에서 Jupyter를 시작하세요. README.md를 확인하세요.')
sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from labkit.common import ensure_data, DATA, output_dir, save_json
ensure_data()

## 파티션 수와 반복 측정

In [ ]:
import time
from labkit.common import spark_session
from pyspark.sql import functions as F
spark=spark_session('week12')
base=spark.range(100000).withColumn('key',F.when(F.col('id')%10<8,0).otherwise(F.col('id')%100))
spark.conf.set('spark.sql.adaptive.enabled','false')
rows=[]
expected=None
for partitions in [2,4,8]:
    spark.conf.set('spark.sql.shuffle.partitions',str(partitions))
    query=base.groupBy('key').count()
    result=sorted((r.key,r['count']) for r in query.collect())
    if expected is None: expected=result
    assert result==expected
    for repeat in range(3):
        start=time.perf_counter(); query.collect()
        rows.append({'partitions':partitions,'repeat':repeat,'seconds':time.perf_counter()-start})
display(pd.DataFrame(rows).groupby('partitions').seconds.agg(['median','min','max']))

## 캐시와 broadcast join 계획

In [ ]:
cached=base.cache(); cached.count()  # 캐시 채움은 warm-up으로 분리
try:
    start=time.perf_counter(); cached.groupBy('key').count().collect()
    print('cached query seconds:',time.perf_counter()-start)
    dimension=spark.range(100).withColumnRenamed('id','key').withColumn('segment',F.col('key')%3)
    joined=cached.join(F.broadcast(dimension),'key')
    joined.explain(mode='formatted')
    assert joined.count()==100000
finally:
    cached.unpersist()
out=output_dir('week12'); pd.DataFrame(rows).to_csv(out/'timings.csv',index=False)
print('Spark UI:',spark.sparkContext.uiWebUrl)
# UI 캡처가 필요하면 다음 stop을 실행하기 전에 확인하세요.
spark.stop()

## 확장 과제 및 제출
Spark UI에서 Shuffle Read/Write·spill·task 편향을 기록하세요. AQE on/off, 일반 join/broadcast를 동일 입력으로 비교하고 warm-up을 제외한 중앙값을 보고하세요. 작은 로컬 데이터의 속도 향상을 보장하지 않습니다.

제출: 실행한 노트북, 결과 표/그림, 변경한 조건과 해석. outputs/weekXX에 저장된 자료를 첨부하세요.

평가 기준: 개념 연결 25%, 실행·재현성 30%, 비교 실험 25%, 해석·한계 20%.